# **Problem Statement**

## **Business Context**

GreenBasket Grocers is an online grocery delivery service headquartered in Austin, Texas, operating across 14 major metropolitan areas in the United States, including Austin, Dallas, Houston, Phoenix, Denver, Atlanta, Charlotte, Nashville, Tampa, Orlando, Columbus, Indianapolis, Kansas City, and Salt Lake City. The company runs a network of 140 dark stores supported by around 3,200 gig-based delivery partners, and fulfills close to 85,000 orders per day. It offers fresh produce, dairy, bakery, and packaged goods through scheduled 2-hour delivery windows, with an average order value of 42 dollars and a customer base of over 4 million registered users. GreenBasket has positioned itself as a convenient, reliable alternative to traditional grocery shopping, competing on delivery speed, product freshness, and slot availability.

A recurring operational challenge for the business is failed deliveries. On any given day, roughly 8 to 10 percent of orders either arrive late, get refused by the customer at the door, or come back undelivered due to reasons such as incorrect addresses, customers being unavailable, damaged perishables, or payment issues at the point of delivery. Each failed delivery costs the company between 12 and 28 dollars when accounting for wasted perishables, delivery partner payouts, reverse logistics, and customer service overhead. At the current failure rate, this translates to daily losses of nearly 150,000 to 200,000 dollars, in addition to the harder-to-measure damage to customer trust and repeat purchase behavior. Today, the operations team treats all orders the same way at dispatch, with no reliable method to identify which orders are at high risk of failing before the delivery is attempted.

If GreenBasket can accurately identify high-risk orders at the time of order placement, the operations team can take targeted actions such as confirmation calls, prepaid payment nudges, priority routing, or slot reassignment for those specific orders. Even a 25 percent reduction in the failure rate would save the company close to 18 million dollars annually, meaningfully cut down on perishable wastage, and improve the on-time delivery experience for millions of customers. It would also allow the company to allocate its customer service and dispatch resources more efficiently, focusing effort where it actually moves the needle rather than spreading it thinly across every order.

## **Objective**

To build a classification model that can predict whether an order is likely to fail delivery at the time of order placement, using features such as customer order history, past failure behavior, delivery address characteristics, order value, payment mode, item mix, chosen time slot, and delivery partner load, enabling the operations team to proactively intervene on high-risk orders through confirmation calls, prepaid nudges, priority routing, or slot reassignment, and helping the company reduce perishable wastage, cut delivery costs, and improve overall customer experience.

## **Data Dictionary**

The dataset, `greenbasket_deliveries.csv`, contains 45,000 order-level records spanning a 45-day operational window across the 14 metropolitan markets served by GreenBasket.

### Identifiers & Metadata

| Column | Data Type | Description |
| --- | --- | --- |
| order_id | object | Unique identifier assigned to each order at the time of placement. |
| order_timestamp | datetime | Date and time when the order was placed. |
| customer_id | object | Unique identifier for the customer placing the order. |

### Customer History

| Column | Data Type | Description |
| --- | --- | --- |
| customer_tenure_days | int | Number of days since the customer registered on the platform. |
| past_order_count | int | Total number of prior orders placed by the customer before this one. |
| past_failure_count | int | Total number of prior orders that failed delivery for this customer. |
| past_failure_rate | float | Historical failure rate for the customer, smoothed with a Laplace prior. |
| avg_past_order_value_usd | float | Average value of the customer's prior orders in USD. |
| days_since_last_order | float | Number of days since the customer's most recent prior order. |

### Delivery Address

| Column | Data Type | Description |
| --- | --- | --- |
| delivery_city | object | City where the order is to be delivered. |
| delivery_pin_code | object | Postal code (ZIP) of the delivery location. |
| is_new_address | int | Flag indicating whether this is the first time delivering to this address for the customer. |
| address_type | object | Type of address (apartment, house, office, other). |
| floor_number | int | Floor number, applicable only for apartment deliveries. |
| distance_from_darkstore_km | float | Road distance from the fulfillment dark store to the delivery address. |
| pin_code_failure_rate | float | Historical failure rate at the delivery pin code, smoothed with a Laplace prior. |

### Order Details

| Column | Data Type | Description |
| --- | --- | --- |
| order_value_usd | float | Total value of the order in USD. |
| num_items | int | Total number of items in the order. |
| num_perishable_items | int | Number of perishable items in the order. |
| num_frozen_items | int | Number of frozen items in the order. |
| has_alcohol | int | Flag indicating whether the order contains alcohol, requiring age verification. |
| payment_mode | object | Payment method (prepaid_card, prepaid_wallet, cash_on_delivery). |
| discount_applied_pct | float | Percentage discount applied to the order. |

### Slot & Time

| Column | Data Type | Description |
| --- | --- | --- |
| delivery_slot | object | Chosen delivery window (morning, afternoon, evening, night). |
| day_of_week | object | Day of the week the order was placed. |
| is_weekend | int | Flag indicating whether the order was placed on a weekend. |
| lead_time_hours | float | Time in hours between order placement and the scheduled delivery slot. |

### Operational

| Column | Data Type | Description |
| --- | --- | --- |
| partner_current_load | float | Number of orders currently assigned to the delivery partner in the same slot. |
| darkstore_utilization_pct | float | Fulfillment capacity utilization of the originating dark store at the time of order. |
| weather_condition | object | Weather at the delivery location (clear, cloudy, rain, storm, snow). |
| temperature_f | float | Outside temperature at the delivery location in degrees Fahrenheit. |

### Target

| Column | Data Type | Description |
| --- | --- | --- |
| delivery_failed | int | Binary target variable indicating whether the delivery failed (1) or was successful (0). |

# **Installing and Importing the Necessary Libraries**

In [ ]:
!pip install pandas==2.2.2 numpy==2.0.2 scikit-learn==1.6.1 tensorflow==2.20.0 keras==3.13.2 xgboost==3.2.0 seaborn==0.13.2 matplotlib==3.10.0 -q

**Note**:- After running the above cell, kindly restart the notebook kernel (for VS Code) or runtime (for Google Colab), and run all cells sequentially from the next cell.- On executing the above line of code, you might see a warning regarding package dependencies. This message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# Standard libraries for tracking execution time and vector/matrix operations
import time
import numpy as np

# Data manipulation and visualization libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Tree-based and ensemble machine learning classifiers
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Utilities for handling class imbalance, model evaluation, and metric calculations
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, recall_score, precision_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)

# Deep learning framework and specific layers for building Artificial Neural Networks (ANNs)
import tensorflow as tf
from keras.models import Sequential  # Model for building NN sequentially.
from keras.layers import Dense, Dropout, BatchNormalization

# Preprocessing tools for scaling data and tools for model optimization/explainability
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV  # To tune different models
from sklearn.inspection import permutation_importance

# Configuration settings to suppress warnings and format data display outputs
import warnings
warnings.filterwarnings('ignore')              # Suppress warnings for cleaner output logs
sns.set_style('whitegrid')                     # Set a consistent clean grid style for plots
pd.set_option('display.max_columns', None)     # Prevent truncation of columns when displaying dataframes

# **Loading the Data**

In [ ]:
# uncomment and run the below code snippets if you're using Google Colab and the dataset is present in the Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# **Data Overview**

## Viewing the first and last 5 rows of the dataset

Examine the first and last five rows of the dataset. Identify the available features, observe the values stored in each column, and become familiar with the dataset's structure.

## Checking the shape of the dataset

Identify the total number of rows and columns to understand the size of the dataset before proceeding with the analysis.

## Checking the attribute types

Identify the data type of each feature and determine whether it is numerical, categorical, or datetime. This helps in selecting appropriate preprocessing and analysis techniques.

## Checking the statistical summary

Examine the statistical summary of the numerical features. Observe the key statistics to understand the data distribution and identify any unusual values.

## Checking for missing values

Check for missing values in each feature. Identify which columns contain missing values and the extent of missing data.

## Checking for duplicate values

Check for duplicate records in the dataset. Identify whether any duplicate rows are present before proceeding with the analysis.

# **Data Preprocessing**

- Convert the `order_timestamp` column to a datetime format. This enables time-based analysis and feature engineering.- Extract the day of the week from the `order_timestamp`. This helps capture weekly patterns in the data.- Extract the hour from the `order_timestamp`. This helps capture hourly patterns in the data.

# **Exploratory Data Analysis**

## Utility Functions

In [ ]:
def histogram(data_df, col, title='Histogram', xlabel=None, ylabel='Frequency'):
    plt.figure(figsize=(9, 3.6))
    sns.histplot(data_df[col], bins=50, kde=True)
    plt.title(title)
    plt.xlabel(xlabel if xlabel else col)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()

def barchart(data_df, x_col, y_col=None, title='Bar Chart', xlabel=None, ylabel=None, rot_degrees=30):
    plt.figure(figsize=(10, 6))
    if y_col: # Bivariate bar chart (x vs y)
        sns.barplot(x=x_col, y=y_col, data=data_df)
    else: # Univariate count plot
        order = data_df[x_col].value_counts().index
        sns.countplot(data=data_df, x=x_col, order=order)

    plt.title(title)
    plt.xlabel(xlabel if xlabel else x_col)
    plt.ylabel(ylabel if ylabel else ('Count' if not y_col else y_col))
    plt.xticks(rotation=rot_degrees, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

def boxplot(data_df, x_col, y_col, title='Box Plot', xlabel=None, ylabel=None):
    plt.figure(figsize=(6, 3.8))
    sns.boxplot(data=data_df, x=x_col, y=y_col)
    plt.title(title)
    plt.xlabel(xlabel if xlabel else x_col)
    plt.ylabel(ylabel if ylabel else y_col)
    plt.tight_layout()
    plt.show()

## Univariate Analysis

### `delivery_failed`

### `past_failure_rate`

Generate a histogram of the `past_failure_rate` feature.

Identify the range where most observations are concentrated and check for any skewness or unusual values.

### `pin_code_failure_rate`

Generate a histogram of the `pin_code_failure_rate` feature.

Observe where most pin-code failure rate values are concentrated, assess the spread of the data, and identify any skewness or unusual values.

### `distance_from_darkstore_km`

Generate a histogram of the `distance_from_darkstore_km` feature.

Observe where most delivery distances are concentrated, assess the spread of the data, and identify any skewness or unusual values.

### `order_value_usd`

Generate a histogram of the `order_value_usd` feature.

Observe where most order values are concentrated, assess the spread of the data, and identify any skewness or unusual values.

## Bivariate Analysis

### `delivery_failed` vs `hour`

### `delivery_failed` vs `payment_mode`

Perform the following steps to analyze how failure rates vary across the different payment modes:

1. Calculate the failure rate for each payment mode.

2. Convert the failure rate into percentage values for easier interpretation.

3. Generate a bar chart to visualize the failure rates by payment mode.

4. Identify the payment modes with the highest and lowest failure rates, and observe any patterns related to how customers pay.

### `past_failure_rate` vs `delivery_failed`

Generate a box plot of `past_failure_rate` across the `delivery_failed` classes.

 Compare the distribution of the customer's historical failure rate between successful and failed deliveries, and identify any differences in central tendency, spread, and potential outliers.

### `distance_from_darkstore_km` vs `delivery_failed`

Generate a box plot of `distance_from_darkstore_km` across the `delivery_failed` classes.

Compare the distribution of delivery distances between successful and failed deliveries, and identify any differences in central tendency, spread, and potential outliers.

## Multivariate Analysis

### Correlation Analysis

Create a list of all numerical features that will be used for correlation analysis. These features will be used to examine relationships between numerical variables and identify patterns that may be useful for further analysis and model building.

Generate a correlation matrix for the numerical features. Examine the strength and direction of relationships between variables, identify highly correlated feature pairs, and look for potential multicollinearity that may affect model performance.

# **Data Preprocessing**

## Splitting the data into train, validation, and test sets

This data is **order-level time-series**: 45,000 orders placed sequentially across a 45-day operational window, and delivery-failure patterns evolve over time as customer behavior, operational load, and weather conditions shift. A random split would scatter orders from the same day (and even from the same partner or dark store) across train and test, letting the model implicitly learn short-term patterns it is later scored on - inflating every metric. To evaluate the model the way it would be used in production - predicting failures for future orders based on past data - we split the data chronologically.

The categorical columns need to be one-hot encoded before they can be fed to the tree ensembles and the neural network. We fit the encoding on training data and align the validation and test sets to the same set of columns.

## Missing Value Treatment

### Display the percentage of missing values

Check the percentage of missing values in the training, validation, and test datasets. Identify the features with missing values before selecting an appropriate imputation strategy.

### `avg_past_order_value_usd`

Perform the following steps to handle missing values in the `avg_past_order_value_usd` feature:

1. Choose an imputation strategy (`mean`, `median`, or `mode`) based on the distribution of the feature.

2. Calculate the selected statistic using only the training data.

3. Use the calculated value to fill the missing values in the training, validation, and test datasets.

4. Apply the same imputation value across all three datasets to ensure a consistent preprocessing strategy and prevent data leakage.

### `days_since_last_order`

Perform the following steps to handle missing values in the `days_since_last_order` feature:

1. Choose an imputation strategy (`mean`, `median`, or `mode`) based on the distribution of the feature.

2. Calculate the selected statistic using only the training data.

3. Use the calculated value to fill the missing values in the training, validation, and test datasets.

4. Apply the same imputation value across all three datasets to ensure a consistent preprocessing strategy and prevent data leakage.

### `partner_current_load`

Perform the following steps to handle missing values in the `partner_current_load` feature:

1. Choose an imputation strategy (`mean`, `median`, or `mode`) based on the distribution of the feature.

2. Calculate the selected statistic using only the training data.

3. Use the calculated value to fill the missing values in the training, validation, and test datasets.

4. Apply the same imputation value across all three datasets to ensure a consistent preprocessing strategy and prevent data leakage.

### `darkstore_utilization_pct`

Perform the following steps to handle missing values in the `darkstore_utilization_pct` feature:

1. Choose an imputation strategy (`mean`, `median`, or `mode`) based on the distribution of the feature.

2. Calculate the selected statistic using only the training data.

3. Use the calculated value to fill the missing values in the training, validation, and test datasets.

4. Apply the same imputation value across all three datasets to ensure a consistent preprocessing strategy and prevent data leakage.

### `temperature_f`

Perform the following steps to handle missing values in the `temperature_f` feature:

1. Choose an imputation strategy (`mean`, `median`, or `mode`) based on the distribution of the feature.

2. Calculate the selected statistic using only the training data.

3. Use the calculated value to fill the missing values in the training, validation, and test datasets.

4. Apply the same imputation value across all three datasets to ensure a consistent preprocessing strategy and prevent data leakage.

# **Model Building**

**Note**: All five models provided in the subsequent subsections need to be built and evaluated.

## Model Evaluation Criterion

## Utility Functions

Before moving ahead, we define a function to check the performance of the model using different metrics.- We will be using metric functions defined in sklearn for accuracy, precision, recall, f1_score- We will create a function which will print out all the above metrics in one go.

In [ ]:
def model_performance_classification(model, predictors, target):
    """
    Function to compute different metrics to check classification model performance

    model: classifier (sklearn or tensorflow)
    predictors: independent variables
    target: dependent variable
    """

    # 1. Get raw predictions
    pred_raw = model.predict(predictors)

    # 2. Check the shape or values to handle DL vs ML models safely
    # If the predictions are probabilities (floats between 0 and 1, not exactly 0 or 1)
    # We check if any value falls strictly between 0 and 1
    if np.issubdtype(pred_raw.dtype, np.floating) and not np.all(np.isin(pred_raw, [0.0, 1.0])):
        # FOR TENSORFLOW DL MODELS: threshold the probabilities at 0.5
        pred = (pred_raw > 0.5).astype(int)
    else:
        # FOR SKLEARN ML MODELS: use them directly
        pred = pred_raw

    # Flatten predictions to ensure they match target shape perfectly
    pred = np.array(pred).flatten()

    # 3. Compute metrics
    acc = accuracy_score(target, pred)
    recall = recall_score(target, pred)
    precision = precision_score(target, pred)
    f1 = f1_score(target, pred)

    # 4. Creating a dataframe of metrics
    data_perf = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1": f1},
        index=[0],
    )

    return data_perf

In [ ]:
def plot_confusion_matrix(model, predictors, target):
    """
    To plot the confusion_matrix with percentages

    model: classifier
    predictors: independent variables
    target: dependent variable
    """
    # 1. Get raw predictions
    y_pred_raw = model.predict(predictors)

    # 2. Check if predictions are probabilities (typical for Keras/TF models or some ML models with predict_proba)
    # If y_pred_raw contains float values between 0 and 1, it's likely probabilities.
    if np.issubdtype(y_pred_raw.dtype, np.floating) and np.any((y_pred_raw > 0) & (y_pred_raw < 1)):
        # Binarize probabilities using a threshold (e.g., 0.5)
        y_pred = (y_pred_raw > 0.5).astype(int)
    else:
        # Use predictions directly (typical for scikit-learn classifiers that return binary labels)
        y_pred = y_pred_raw

    # Ensure y_pred is flattened to a 1D array, as confusion_matrix expects this format.
    y_pred = np.array(y_pred).flatten()

    cm = confusion_matrix(target, y_pred)
    labels = np.asarray(
        [
            ["{0:0.0f}".format(item) + "\n{0:.2%}".format(item / cm.flatten().sum())]
            for item in cm.flatten()
        ]
    ).reshape(2, 2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=labels, fmt="")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.show() # Added plt.show() to explicitly display the plot

## Decision Tree Classifier

Perform the following steps to train and evaluate the Decision Tree model:

1. Create a Decision Tree classifier using `class_weight='balanced'` to assign higher importance to the minority class and reduce the impact of class imbalance.

2. Train the model using the training dataset.

3. Evaluate the model on the training dataset by generating a confusion matrix and computing the performance metrics.

4. Evaluate the model on the validation dataset by generating a confusion matrix and computing the performance metrics.

5. Compare the training and validation results to assess how well the model generalizes and identify any signs of overfitting or underfitting.

## Random Forest Classifier

Perform the following steps to train and evaluate the Random Forest model:

1. Create a Random Forest classifier using `class_weight='balanced'` to assign higher importance to the minority class and reduce the impact of class imbalance.

2. Train the model using the training dataset.

3. Evaluate the model on the training dataset by generating a confusion matrix and computing the performance metrics.

4. Evaluate the model on the validation dataset by generating a confusion matrix and computing the performance metrics.

5. Compare the training and validation results to assess how well the model generalizes and identify any signs of overfitting or underfitting.

## Gradient Boosting Classifier

Perform the following steps to train and evaluate the Gradient Boosting model:

1. Create a Gradient Boosting classifier using the selected random state.

2. Train the model using the training dataset.

3. Evaluate the model on the training dataset by generating a confusion matrix and computing the performance metrics.

4. Evaluate the model on the validation dataset by generating a confusion matrix and computing the performance metrics.

5. Compare the training and validation results to assess how well the model generalizes and identify any signs of overfitting or underfitting.

## XGBoost

Perform the following steps to train and evaluate the XGBoost model:

1. Calculate the number of negative and positive samples in the training data.

2. Compute the `scale_pos_weight` value as the ratio of negative samples to positive samples to address class imbalance.

3. Create an XGBoost classifier using the calculated `scale_pos_weight` and the selected random state.

4. Train the model using the training dataset.

5. Evaluate the model on the training dataset by generating a confusion matrix and computing the performance metrics.

6. Evaluate the model on the validation dataset by generating a confusion matrix and computing the performance metrics.

7. Compare the training and validation results to assess how well the model generalizes and identify any signs of overfitting or underfitting.

## Neural Networks(ANN)

In [ ]:
tf.keras.backend.clear_session()

Perform the following steps to standardize the feature values:

1. Create a `StandardScaler` object to standardize the numerical features.

2. Fit the scaler using only the training dataset and transform the training features.

3. Use the same fitted scaler to transform the validation and test datasets.

4. Apply the same scaling parameters across all datasets to ensure a consistent preprocessing strategy and prevent data leakage.

Perform the following steps to build the neural network architecture:

1. Choose the number of neurons for the first hidden layer between **8 and 128**.   - Fewer neurons (**8–32**) create a simpler model that trains faster and is less likely to overfit.   - More neurons (**64–128**) enable the model to learn more complex patterns but increase training time and the risk of overfitting.

2. Choose an activation function for the first hidden layer (for example, `relu`, `tanh`, or `sigmoid`) based on the learning behavior you want.

3. Create the output layer with a single neuron and a `sigmoid` activation function to predict the probability of the positive class.

Display the neural network architecture.

Review the number of layers, the output shape of each layer, and the total number of trainable parameters before training the model.

Perform the following steps to configure the neural network for training:

1. Choose an optimizer (`sgd` or `adam`) based on the desired training behavior.   - `sgd` updates the model using simple gradient descent and may require more epochs to converge.   - `adam` automatically adapts the learning rate, often converges faster, and is a common default choice.

2. Use `binary_crossentropy` as the loss function since this is a binary classification problem.

3. Configure the model to track **Recall**, **Precision**, and **Binary Accuracy** during training to evaluate its classification performance.

Perform the following steps to configure the training process:

1. Choose the number of training epochs between **10 and 100** based on the desired training duration and model performance.

2. Select a batch size between **16 and 128** based on the available computational resources and training behavior.

3. Use the selected values to control how long the model trains and how many samples are processed in each weight update.

In [ ]:
weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: weights[i] for i in range(len(weights))}

In [ ]:
start = time.time()
history = model.fit(X_train_scaled, y_train, validation_data=(X_valid_scaled,y_valid) , class_weight=class_weight_dict ,batch_size=batch_size, epochs=epochs)
end = time.time()

In [ ]:
print("Time taken in seconds ",end-start)

Perform the following steps to evaluate the neural network model:

1. Evaluate the model on the training dataset by generating a confusion matrix and computing the performance metrics.

2. Evaluate the model on the validation dataset by generating a confusion matrix and computing the performance metrics.

3. Compare the training and validation results to assess how well the model generalizes and identify any signs of overfitting or underfitting.

## **Baseline Model Performance Comparison**

### Training performance comparison

Perform the following steps to compare the training performance of all baseline models:

1. Combine the training performance metrics of all baseline models into a single comparison table.

2. Display the consolidated table to compare the performance of each model across the selected evaluation metrics.

3. Use the comparison to identify the best-performing models for hyperparameter tuning.

### Validation performance comparison

Perform the following steps to compare the validation performance of all baseline models:

1. Combine the validation performance metrics of all baseline models into a single comparison table.

2. Display the consolidated table to compare the performance of each model across the selected evaluation metrics.

3. Use the comparison to identify the best-performing models for hyperparameter tuning based on their validation performance.

# **Hyperparameter Tuning**

**Note:** Choose at least two best-performing baseline models across the train and validation sets to proceed with tuning.

## XGBoost

Perform the following steps to define the hyperparameter search space for the XGBoost model:

1. Choose values for **`n_estimators`**, which controls the number of trees in the model. More trees can improve learning but increase training time.

2. Choose values for **`learning_rate`**, which controls how quickly the model learns. Smaller values require more trees, while larger values learn faster but may overshoot.

3. Choose values for **`max_depth`**, which controls the maximum depth of each tree. Deeper trees capture more complex patterns but are more likely to overfit.

4. Choose values for **`min_child_weight`**, which controls the minimum weight required to create a new split. Larger values produce more conservative trees.

5. Choose values for **`gamma`**, which specifies the minimum loss reduction required before making a split. Larger values reduce unnecessary splits.

6. Choose values for **`subsample`**, which determines the fraction of training samples used to build each tree. Lower values improve generalization, while higher values use more data.

7. Choose values for **`colsample_bytree`**, which specifies the fraction of features used to build each tree. Lower values increase diversity among trees.

8. Choose values for **`reg_alpha`** and **`reg_lambda`**, which apply L1 and L2 regularization to reduce overfitting.

9. Use the selected values to create the parameter grid for hyperparameter tuning and identify the best-performing model configuration.

Perform the following steps to tune the XGBoost model using randomized search:

1. Perform randomized search with **5-fold cross-validation** using the selected evaluation metric.

2. Train the model on each sampled hyperparameter combination and evaluate its cross-validation performance.

3. Display the best hyperparameter combination and its corresponding cross-validation score.

1. Retrieve the best-performing XGBoost model identified during the randomized search.

2. Retrain the selected model using the complete training dataset.

3. Use the retrained model for subsequent evaluation on the training, validation, and test datasets.

## Neural Networks

In [ ]:
tf.keras.backend.clear_session()

Perform the following steps to build the tuned neural network architecture:

1. Choose the number of neurons and an activation function for the first hidden layer. Then apply **Batch Normalization** to stabilize training, followed by **Dropout** to reduce overfitting.

2. Choose the number of neurons and an activation function for the second hidden layer. Again, apply **Batch Normalization** followed by **Dropout** to improve generalization.

3. Create the output layer with a single neuron and a **`sigmoid`** activation function to predict the probability of the positive class.

4. Review the selected architecture before proceeding with model training.

Display the tuned neural network architecture.

Review the number of layers, the output shape of each layer, and the total number of trainable parameters before training the model.

Perform the following steps to configure the tuned neural network for training:

1. Choose an optimizer (`adam` or `sgd`) based on the desired training behavior.   - `adam` automatically adapts the learning rate, often converges faster, and is a common default choice.   - `sgd` updates the model using simple gradient descent and may require more epochs to converge.

2. Use `binary_crossentropy` as the loss function since this is a binary classification problem.

3. Configure the model to track **Recall**, **Precision**, and **Binary Accuracy** during training to evaluate its classification performance.

Perform the following steps to configure the training process:

1. Choose the number of training epochs between **20 and 150** based on the desired training duration and model performance.

2. Select a batch size between **16 and 128** based on the available computational resources and training behavior.

3. Use the selected values to control how long the model trains and how many samples are processed in each weight update.

In [ ]:
start = time.time()
history = model1.fit(X_train_scaled, y_train, validation_data=(X_valid_scaled,y_valid) ,class_weight=class_weight_dict, batch_size=batch_size, epochs=epochs)
end = time.time()

In [ ]:
print("Time taken in seconds ",end-start)

Perform the following steps to evaluate the tuned neural network model:

1. Evaluate the model on the training dataset by generating a confusion matrix and computing the performance metrics.

2. Evaluate the model on the validation dataset by generating a confusion matrix and computing the performance metrics.

3. Compare the training and validation results to assess whether the tuned architecture improves generalization and reduces overfitting compared to the baseline neural network.

## Decision Tree

Perform the following steps to define the hyperparameter search space for the Decision Tree model:

1. Choose the splitting **`criterion`** (`gini` or `entropy`) to determine how the model selects the best feature at each split.

2. Choose values for **`max_depth`** to control the maximum depth of the tree. Deeper trees can capture more complex patterns but are more likely to overfit.

3. Choose values for **`min_samples_split`** to control the minimum number of samples required to split a node. Larger values create more conservative trees.

4. Choose values for **`min_samples_leaf`** to control the minimum number of samples required in each leaf node. Larger values produce smoother and more generalized trees.

5. Choose the **`max_features`** strategy (`sqrt` or `log2`) to control how many features are considered when searching for the best split.

6. Use the selected values to create the parameter grid for hyperparameter tuning and identify the best-performing model configuration.

Perform the following steps to tune the Decision Tree model using randomized search:

1. Perform randomized search with **5-fold cross-validation** using the selected evaluation metric.

2. Train the model on each sampled hyperparameter combination and evaluate its cross-validation performance.

3. Display the best hyperparameter combination and its corresponding cross-validation score.

Display the best Decision Tree model identified during hyperparameter tuning.

Review the optimized model configuration before using it for retraining and evaluation.

Evaluate the tuned Decision Tree model on the training dataset and display its performance metrics.

Use the results to assess how well the tuned model has learned the training data.

Evaluate the tuned Decision Tree model on the validation dataset and display its performance metrics.

Compare these results with the training performance to assess how well the tuned model generalizes to unseen data.

## Random Forest

Perform the following steps to define the hyperparameter search space for the Random Forest model:

1. Choose values for **`n_estimators`**, which controls the number of trees in the forest. More trees generally improve stability but increase training time.

2. Choose values for **`max_depth`** to control the maximum depth of each tree. Deeper trees capture more complex patterns but are more likely to overfit.

3. Choose values for **`min_samples_split`** to control the minimum number of samples required to split a node. Larger values create more conservative trees.

4. Choose values for **`min_samples_leaf`** to control the minimum number of samples required in each leaf node. Larger values improve generalization by preventing overly specific splits.

5. Choose the **`max_features`** strategy (`sqrt` or `log2`) to control how many features are considered when searching for the best split in each tree.

6. Use the selected values to create the parameter grid for hyperparameter tuning and identify the best-performing model configuration.

Perform the following steps to tune the Random Forest model using randomized search:

1. Perform randomized search with **5-fold cross-validation** using the selected evaluation metric.

2. Train the model on each sampled hyperparameter combination and evaluate its cross-validation performance.

3. Display the best hyperparameter combination and its corresponding cross-validation score.

Perform the following steps after hyperparameter tuning:

1. Retrieve the best-performing Random Forest model identified during the randomized search.

2. Retrain the selected model using the complete training dataset.

3. Evaluate the retrained model on the training dataset and display its performance metrics.

4. Evaluate the retrained model on the validation dataset and display its performance metrics.

5. Compare the training and validation results to assess how well the tuned model generalizes to unseen data.

## Gradient Boosting

Perform the following steps to define the hyperparameter search space for the Gradient Boosting model:

1. Choose values for **`n_estimators`**, which controls the number of boosting stages. More boosting stages can improve learning but increase training time.

2. Choose values for **`learning_rate`**, which controls how quickly the model learns. Smaller values require more boosting stages, while larger values learn faster but may overfit.

3. Choose values for **`max_depth`** to control the maximum depth of each decision tree. Deeper trees capture more complex patterns but are more likely to overfit.

4. Choose values for **`subsample`**, which determines the fraction of training samples used for each boosting stage. Lower values improve generalization by introducing randomness, while higher values use more data for learning.

5. Choose the **`max_features`** strategy (`sqrt` or `log2`) to control how many features are considered when searching for the best split in each tree.

6. Use the selected values to create the parameter grid for hyperparameter tuning and identify the best-performing model configuration.

Perform the following steps to tune the Gradient Boosting model using randomized search:

1. Perform randomized search with **3-fold cross-validation** using the selected evaluation metric.

2. Train the model on each sampled hyperparameter combination and evaluate its cross-validation performance.

3. Display the best hyperparameter combination and its corresponding cross-validation score.

Perform the following steps after hyperparameter tuning:

1. Retrieve the best-performing Gradient Boosting model identified during the randomized search.

2. Retrain the selected model using the complete training dataset.

3. Evaluate the retrained model on the training dataset and display its performance metrics.

4. Evaluate the retrained model on the validation dataset and display its performance metrics.

5. Compare the training and validation results to assess how well the tuned model generalizes to unseen data.

# **Final Model Selection**

Perform the following steps to compare the training performance of the tuned models:

1. Add the training performance metrics of the tuned models to the existing comparison table.

2. Display the updated comparison table.

3. Compare the baseline and tuned models to determine whether hyperparameter tuning improved the training performance.

Perform the following steps to compare the validation performance of the tuned models:

1. Add the validation performance metrics of the tuned models to the existing comparison table.

2. Display the updated comparison table.

3. Compare the baseline and tuned models to determine whether hyperparameter tuning improved the validation performance and generalization.

Perform the following steps to select the final model:

1. Review the performance of all baseline and tuned models using the selected evaluation metric.

2. Select the model that best meets the business objective and demonstrates strong generalization on the validation dataset.

3. Assign the selected model as the final model for evaluation on the test dataset.

## Feature Importance

In [ ]:
def plot_feature_importances(model, X, y, feature_names=None, color="violet", figsize=(10, 10)):
    """
    Plots feature importances for both Traditional ML models (XGBoost, RF)
    and Deep Learning models (ANNs).

    Parameters:
    - model: Trained model object (XGBoost, Sklearn, Keras wrapper, etc.)
    - X: Evaluation features (DataFrame or 2D array)
    - y: Evaluation targets (Series or 1D array)
    - feature_names: List of feature names. If None and X is a DataFrame, uses X.columns.
    """
    # 1. Automatically grab feature names if not provided
    if feature_names is None:
        if hasattr(X, 'columns'):
            feature_names = X.columns
        else:
            feature_names = [f"Feature {i}" for i in range(X.shape[1])]

    # Convert X to numpy if it's a DataFrame for permutation calculation safety
    X_val = X.values if hasattr(X, 'columns') else X

    # 2. Extract or calculate importances
    if hasattr(model, 'feature_importances_'):
        print("Detected Tree-based model. Extracting built-in feature importances...")
        importances = model.feature_importances_
        title = "Feature Importances (Tree-based Model)"
        xlabel = "Relative Importance"
    else:
        print("Detected ANN/Black-box model. Calculating Permutation Importance...")
        # n_repeats is how many times a feature is shuffled; higher is more accurate but slower
        result = permutation_importance(model, X_val, y, n_repeats=5, random_state=42)
        importances = result.importances_mean
        title = "Feature Importances (Permutation Importance - ANN)"
        xlabel = "Mean Performance Drop when Shuffled"

    # 3. Sort the importances in ascending order
    indices = np.argsort(importances)

    # 4. Plotting
    plt.figure(figsize=figsize)
    plt.title(title)
    plt.barh(range(len(indices)), importances[indices], color=color, align="center")
    plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
    plt.xlabel(xlabel)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_feature_importances(best_model, X_train, y_train)

## Final Model Test Performance

1. Evaluate the selected model using the unseen test dataset.

2. Display the performance metrics and confusion matrix to measure how well the model generalizes to new data.

3. Use the test results as the final estimate of the model's real-world performance.

# **Business Insights and Recommendations**

## Business Insights

-
-
-

## Recommendations

-
-
-